# 💰 Personal Finance AI Agent (LangChain)

**Objective:** Build an AI-powered personal finance assistant using LangChain agents and custom tools. The LLM decides which tool to use for each query, and the calculations are done by the tools, not by the LLM.

**Tools built:**
1. Calculator Tool
2. EMI Calculator Tool
3. SIP Investment Calculator Tool
4. Budget Planner Tool (50-30-20 rule)

**Note on the LLM:** This project uses **Google Gemini** instead of OpenAI GPT-4o-mini, because Gemini offers a free API tier. The agent logic is identical for any chat model that supports tool calling.

## 1. Setup

In [1]:
# Run once if packages are missing:
# !pip install langchain langchain-google-genai python-dotenv

In [2]:
import os
import warnings
from getpass import getpass
from dotenv import load_dotenv
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")
load_dotenv()

# Reads GOOGLE_API_KEY from a .env file; asks for it if not found (never hardcode your key)
if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google API key: ")

print("API key loaded")

API key loaded


## 2. Build the Tools

In [3]:
from langchain.tools import tool

@tool
def calculator_tool(expression: str) -> str:
    """Use this tool for general mathematical calculations.
    Input should be a valid math expression, e.g. '0.20*85000' or '5000*12'."""
    try:
        expression = expression.replace("%", "/100*")
        result = eval(expression, {"__builtins__": {}})
        return f"Result: {result}"
    except Exception as e:
        return f"Error calculating: {e}"

In [4]:
@tool
def emi_calculator_tool(loan_amount: float, annual_rate: float, years: int) -> str:
    """Use this tool to calculate loan EMI (Equated Monthly Installment).
    Inputs: loan_amount (principal in rupees), annual_rate (annual interest rate in %), years (loan tenure in years).
    Example: loan_amount=1000000, annual_rate=9, years=20"""
    try:
        P = loan_amount
        r = (annual_rate / 12) / 100   # monthly interest rate
        n = years * 12                 # total months
        emi = P * r * (1 + r) ** n / ((1 + r) ** n - 1)
        return (f"For a loan of ₹{P:,.0f} at {annual_rate}% annual interest for {years} years:\n"
                f"Monthly EMI = ₹{emi:,.2f}\n"
                f"Total Payment = ₹{emi * n:,.2f}\n"
                f"Total Interest = ₹{(emi * n) - P:,.2f}")
    except Exception as e:
        return f"Error calculating EMI: {e}"

In [5]:
@tool
def sip_calculator_tool(monthly_investment: float, annual_return: float, years: int) -> str:
    """Use this tool to calculate the future value of a SIP (Systematic Investment Plan).
    Inputs: monthly_investment (amount invested per month in rupees), annual_return (expected annual return in %), years (investment duration in years).
    Example: monthly_investment=5000, annual_return=12, years=10"""
    try:
        P = monthly_investment
        r = (annual_return / 12) / 100  # monthly return rate
        n = years * 12                  # total months
        fv = P * (((1 + r) ** n - 1) / r)
        total_invested = P * n
        return (f"For a monthly SIP of ₹{P:,.0f} at {annual_return}% annual return for {years} years:\n"
                f"Future Value = ₹{fv:,.2f}\n"
                f"Total Invested = ₹{total_invested:,.2f}\n"
                f"Total Gains = ₹{fv - total_invested:,.2f}")
    except Exception as e:
        return f"Error calculating SIP: {e}"

In [6]:
@tool
def budget_planner_tool(monthly_income: float) -> str:
    """Use this tool to suggest a monthly budget allocation using the 50-30-20 rule.
    Input: monthly_income (total monthly income in rupees).
    Example: monthly_income=80000"""
    try:
        return (f"Suggested budget allocation for monthly income of ₹{monthly_income:,.0f}:\n"
                f"Needs (50%) = ₹{monthly_income * 0.50:,.2f} — rent, groceries, bills, essentials\n"
                f"Wants (30%) = ₹{monthly_income * 0.30:,.2f} — entertainment, dining out, shopping\n"
                f"Savings (20%) = ₹{monthly_income * 0.20:,.2f} — investments, emergency fund")
    except Exception as e:
        return f"Error creating budget: {e}"

## 3. Test Each Tool Individually
Before adding the agent, we check that each tool gives correct results on its own.

In [7]:
print(calculator_tool.invoke({"expression": "0.20*85000"}))
print()
print(emi_calculator_tool.invoke({"loan_amount": 1000000, "annual_rate": 9, "years": 20}))
print()
print(sip_calculator_tool.invoke({"monthly_investment": 5000, "annual_return": 12, "years": 10}))
print()
print(budget_planner_tool.invoke({"monthly_income": 80000}))

Result: 17000.0

For a loan of ₹1,000,000 at 9.0% annual interest for 20 years:
Monthly EMI = ₹8,997.26
Total Payment = ₹2,159,342.29
Total Interest = ₹1,159,342.29

For a monthly SIP of ₹5,000 at 12.0% annual return for 10 years:
Future Value = ₹1,150,193.45
Total Invested = ₹600,000.00
Total Gains = ₹550,193.45

Suggested budget allocation for monthly income of ₹80,000:
Needs (50%) = ₹40,000.00 — rent, groceries, bills, essentials
Wants (30%) = ₹24,000.00 — entertainment, dining out, shopping
Savings (20%) = ₹16,000.00 — investments, emergency fund


## 4. Create the Agent
The agent uses Gemini as its LLM and automatically decides which tool to call based on the user's query.

In [11]:
import time
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
)

tools = [calculator_tool, emi_calculator_tool, sip_calculator_tool, budget_planner_tool]
agent = create_agent(llm, tools)


def ask_agent(query: str) -> str:
    """Send a query to the agent and return the answer plus which tools were used.
    Retries automatically if Google's servers are temporarily overloaded (503 error)."""
    max_retries = 3
    for attempt in range(max_retries):
        try:
            result = agent.invoke({"messages": [{"role": "user", "content": query}]})
            break
        except Exception as e:
            if "503" in str(e) or "UNAVAILABLE" in str(e):
                if attempt < max_retries - 1:
                    time.sleep(3)
                    continue
                else:
                    return "⚠️ The AI service is temporarily busy. Please try again in a moment."
            else:
                return f"⚠️ An error occurred: {e}"

    tool_calls_made = []
    for msg in result["messages"]:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                tool_calls_made.append(tc["name"])

    content = result["messages"][-1].content
    if isinstance(content, list):
        text = "".join(b.get("text", "") for b in content if isinstance(b, dict))
    else:
        text = content

    tools_info = f"\n\n**[Tools used: {', '.join(tool_calls_made) if tool_calls_made else 'None (LLM answered directly)'}]**"
    return text + tools_info

## 5. Demonstrate the Agent with 5 Queries
Each answer shows which tool the agent selected.

In [12]:
queries = [
    "Calculate EMI for a 10 lakh loan at 8% interest for 20 years",
    "If I invest ₹7000 monthly for 15 years at 10% return, how much will I have?",
    "Create a budget plan for income ₹90,000",
    "What is 20% of ₹85,000",
    "How much will I get if I invest ₹3000 per month for 5 years at 8% return",
]

for i, q in enumerate(queries, 1):
    display(Markdown(f"### Query {i}: {q}"))
    display(Markdown(ask_agent(q)))
    display(Markdown("---"))
    

### Query 1: Calculate EMI for a 10 lakh loan at 8% interest for 20 years

⚠️ An error occurred: Error calling model 'gemini-3.6-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 17.657198746s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.6-flash'}, 'quotaValue': '20'}]}, {'@type': 'type.googleapis.com/google.rpc.RetryInfo', 'retryDelay': '17s'}]}}

---

### Query 2: If I invest ₹7000 monthly for 15 years at 10% return, how much will I have?

⚠️ An error occurred: Error calling model 'gemini-3.6-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 38.111310347s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.6-flash'}, 'quotaValue': '20'}]}, {'@type': 'type.googleapis.com/google.rpc.RetryInfo', 'retryDelay': '38s'}]}}

...

⚠️ An error occurred: Error calling model 'gemini-3.6-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 21.861520968s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-3.6-flash', 'location': 'global'}, 'quotaValue': '20'}]}, {'@type': 'type.googleapis.com/google.rpc.RetryInfo', 'retryDelay': '21s'}]}}

---

### Query 5: How much will I get if I invest ₹3000 per month for 5 years at 8% return

⚠️ An error occurred: Error calling model 'gemini-3.6-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 45.965232321s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.6-flash'}, 'quotaValue': '20'}]}, {'@type': 'type.googleapis.com/google.rpc.RetryInfo', 'retryDelay': '45s'}]}}

---

## 6. Conclusion

- Built 4 custom tools: Calculator, EMI, SIP and Budget Planner.
- Built a LangChain agent that automatically selects the right tool for each query.
- All 5 test queries were routed to the correct tool, and the calculations came from the tools rather than the LLM.
- The same logic is deployed as a Streamlit web app in `app.py`.